In [3]:
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# nfl_games_2015_2024
df = pd.read_csv('/content/drive/MyDrive/nfl_games_2015_2024.csv')


In [6]:
# preview the table (x rows x 46 columns)
df.head(4)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium


In [7]:
game_type_values = df['game_type'].unique()
print(game_type_values)

['REG' 'WC' 'DIV' 'CON' 'SB']


REG — Regular season game.
WC — Wild Card round (first round of playoffs).
DIV — Divisional round (second round).
CON — Conference Championship (AFC/NFC title game, the round right before the Super Bowl).
SB — Super Bowl

In [8]:
# Filter to keep only super bowl rows where the season is 2017 or greater
sb = df[(df['game_type'] == 'SB') & (df['season'] >= 2017)].copy()

In [9]:
print(len(df))
print(len(sb))

2743
8


In [10]:
# create an id column in the sb table from the calendar year of game
sb['game_year_id'] = pd.to_datetime(sb['gameday']).dt.year

In [11]:
sb.head(3) # x rows and 47 columns

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,game_year_id
800,2017_21_PHI_NE,2017,SB,21,2018-02-04,Sunday,18:31,PHI,41,NE,...,00-0029567,00-0019596,Nick Foles,Tom Brady,Doug Pederson,Bill Belichick,Gene Steratore,MIN01,U.S. Bank Stadium,2018
1067,2018_21_NE_LA,2018,SB,21,2019-02-03,Sunday,18:30,NE,13,LA,...,00-0019596,00-0033106,Tom Brady,Jared Goff,Bill Belichick,Sean McVay,John Parry,ATL97,Mercedes-Benz Stadium,2019
1334,2019_21_SF_KC,2019,SB,21,2020-02-02,Sunday,18:30,SF,20,KC,...,00-0031345,00-0033873,Jimmy Garoppolo,Patrick Mahomes,Kyle Shanahan,Andy Reid,Bill Vinovich,MIA00,Hard Rock Stadium,2020


In [12]:
# Keep columns list
keep_columns = [ 'game_year_id', 'away_team', 'home_team', 'spread_line', 'total_line', 'roof',
    'away_qb_name', 'home_qb_name']

sb_clean = sb[keep_columns].reset_index(drop=True)

In [13]:
sb_clean.head(3) # x rows and 8 columns

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name
0,2018,PHI,NE,4.5,48.5,dome,Nick Foles,Tom Brady
1,2019,NE,LA,-2.0,55.5,closed,Tom Brady,Jared Goff
2,2020,SF,KC,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes


In [14]:
sb_clean

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name
0,2018,PHI,NE,4.5,48.5,dome,Nick Foles,Tom Brady
1,2019,NE,LA,-2.0,55.5,closed,Tom Brady,Jared Goff
2,2020,SF,KC,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes
3,2021,KC,TB,-3.0,55.0,outdoors,Patrick Mahomes,Tom Brady
4,2022,LA,CIN,-4.5,48.5,dome,Matthew Stafford,Joe Burrow
5,2023,KC,PHI,1.0,51.0,closed,Patrick Mahomes,Jalen Hurts
6,2024,SF,KC,-1.5,47.0,dome,Brock Purdy,Patrick Mahomes
7,2025,KC,PHI,-1.5,48.5,dome,Patrick Mahomes,Jalen Hurts


In [22]:
# Code to remove the abbreviations

team_name_map = {
    'PHI': 'Philadelphia Eagles',
    'NE': 'New England Patriots',
    'SF': 'San Francisco 49ers',
    'KC': 'Kansas City Chiefs',
    'LA': 'Los Angeles Rams',
    'TB': 'Tampa Bay Buccaneers',
    'CIN': 'Cincinnati Bengals',
}

# sb_clean['away_team'] = sb_clean['away_team_full']
# #sb_clean['away_team'].map(team_name_map)
# sb_clean['home_team'] = sb_clean['home_team_full']
# #sb_clean['home_team'].map(team_name_map)

# # sanity check — should print 0
# print(sb_clean['away_team_full'].isna().sum())
# print(sb_clean['home_team_full'].isna().sum())

# 1. Clean the team names by stripping hidden spaces
sb_clean['away_team'] = sb_clean['away_team'].astype(str).str.strip()
sb_clean['home_team'] = sb_clean['home_team'].astype(str).str.strip()

# 2. Force the dictionary mapping to overwrite the columns
sb_clean['away_team'] = sb_clean['away_team'].replace(team_name_map)
sb_clean['home_team'] = sb_clean['home_team'].replace(team_name_map)

In [23]:
sb_clean

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name
0,2018,Philadelphia Eagles,New England Patriots,4.5,48.5,dome,Nick Foles,Tom Brady
1,2019,New England Patriots,Los Angeles Rams,-2.0,55.5,closed,Tom Brady,Jared Goff
2,2020,San Francisco 49ers,Kansas City Chiefs,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes
3,2021,Kansas City Chiefs,Tampa Bay Buccaneers,-3.0,55.0,outdoors,Patrick Mahomes,Tom Brady
4,2022,Los Angeles Rams,Cincinnati Bengals,-4.5,48.5,dome,Matthew Stafford,Joe Burrow
5,2023,Kansas City Chiefs,Philadelphia Eagles,1.0,51.0,closed,Patrick Mahomes,Jalen Hurts
6,2024,San Francisco 49ers,Kansas City Chiefs,-1.5,47.0,dome,Brock Purdy,Patrick Mahomes
7,2025,Kansas City Chiefs,Philadelphia Eagles,-1.5,48.5,dome,Patrick Mahomes,Jalen Hurts


In [24]:
# superbowl_games
df_2 = pd.read_csv('/content/drive/MyDrive/superbowl_games.csv')

In [25]:
df_2.head(3)

,Game,Date (Season),Attendance,winner_team,winner_conference_code,winner_appearance_no,winner_wins_to_date,winner_losses_to_date,loser_team,loser_conference_code,loser_appearance_no,loser_wins_to_date,loser_losses_to_date,winner_score,loser_score,venue,venue_host_count,city,city_host_count
0,I,"January 15, 1967 (1966 AFL/1966 NFL)",61946,Green Bay Packers,n,1,1,0,Kansas City Chiefs,a,1,0,1,35,10,Los Angeles Memorial Coliseum,1,"Los Angeles, California",1
1,II,"January 14, 1968 (1967 AFL/1967 NFL)",75546,Green Bay Packers,n,2,2,0,Oakland Raiders,a,1,0,1,33,14,Miami Orange Bowl,1,"Miami, Florida",1
2,III,"January 12, 1969 (1968 AFL/1968 NFL)",75389,New York Jets,a,1,1,0,Baltimore Colts,n,1,0,1,16,7,Miami Orange Bowl,2,"Miami, Florida",2


In [26]:
# Create an id column
df_2['game_year_id'] = pd.to_datetime(df_2['Date (Season)'].str.split('(').str[0].str.strip()).dt.year

# Filter to the target window
sb_2 = df_2[(df_2['game_year_id'] >= 2018) & (df_2['game_year_id'] <= 2026)].copy()

print(len(sb_2))

9


In [27]:
# format for the id column
sb_2

,Game,Date (Season),Attendance,winner_team,winner_conference_code,winner_appearance_no,winner_wins_to_date,winner_losses_to_date,loser_team,loser_conference_code,loser_appearance_no,loser_wins_to_date,loser_losses_to_date,winner_score,loser_score,venue,venue_host_count,city,city_host_count,game_year_id
51,LII,"February 4, 2018 (2017)",67612,Philadelphia Eagles,N,3,1,2,New England Patriots,A,10,5,5,41,33,U.S. Bank Stadium,1,"Minneapolis, Minnesota",2,2018
52,LIII,"February 3, 2019 (2018)",70081,New England Patriots,A,11,6,5,Los Angeles Rams,N,4,1,3,13,3,Mercedes-Benz Stadium,1,"Atlanta, Georgia",3,2019
53,LIV,"February 2, 2020 (2019)",62417,Kansas City Chiefs,A,3,2,1,San Francisco 49ers,N,7,5,2,31,20,Hard Rock Stadium,6,"Miami Gardens, Florida",11,2020
54,LV,"February 7, 2021 (2020)",24835,Tampa Bay Buccaneers,N,2,2,0,Kansas City Chiefs,A,4,2,2,31,9,Raymond James Stadium,3,"Tampa, Florida",5,2021
55,LVI,"February 13, 2022 (2021)",70048,Los Angeles Rams,N,5,2,3,Cincinnati Bengals,A,3,0,3,23,20,SoFi Stadium,1,"Inglewood, California",8,2022
56,LVII,"February 12, 2023 (2022)",67827,Kansas City Chiefs,A,5,3,2,Philadelphia Eagles,N,4,1,3,38,35,State Farm Stadium,3,"Glendale, Arizona",4,2023
57,LVIII,"February 11, 2024 (2023)",61629,Kansas City Chiefs,A,6,4,2,San Francisco 49ers,N,8,5,3,25,22 (OT),Allegiant Stadium,1,"Paradise, Nevada",1,2024
58,LIX,"February 9, 2025 (2024)",65719,Philadelphia Eagles,N,5,2,3,Kansas City Chiefs,A,7,4,3,40,22,Caesars Superdome,8,"New Orleans, Louisiana",11,2025
59,LX,"February 8, 2026 (2025)",70823,Seattle Seahawks,N,4,2,2,New England Patriots,A,12,6,6,29,13,Levi's Stadium,2,"Santa Clara, California",3,2026


In [28]:
keep_columns_2 = [
    'game_year_id', 'Attendance',
    'winner_team', 'loser_team',
    'winner_score', 'loser_score',
    'winner_appearance_no', 'winner_wins_to_date', 'winner_losses_to_date',
    'loser_appearance_no', 'loser_wins_to_date', 'loser_losses_to_date',
    'venue', 'city'
]

sb_2_clean = sb_2[keep_columns_2].reset_index(drop=True)

sb_2_clean

,game_year_id,Attendance,winner_team,loser_team,winner_score,loser_score,winner_appearance_no,winner_wins_to_date,winner_losses_to_date,loser_appearance_no,loser_wins_to_date,loser_losses_to_date,venue,city
0,2018,67612,Philadelphia Eagles,New England Patriots,41,33,3,1,2,10,5,5,U.S. Bank Stadium,"Minneapolis, Minnesota"
1,2019,70081,New England Patriots,Los Angeles Rams,13,3,11,6,5,4,1,3,Mercedes-Benz Stadium,"Atlanta, Georgia"
2,2020,62417,Kansas City Chiefs,San Francisco 49ers,31,20,3,2,1,7,5,2,Hard Rock Stadium,"Miami Gardens, Florida"
3,2021,24835,Tampa Bay Buccaneers,Kansas City Chiefs,31,9,2,2,0,4,2,2,Raymond James Stadium,"Tampa, Florida"
4,2022,70048,Los Angeles Rams,Cincinnati Bengals,23,20,5,2,3,3,0,3,SoFi Stadium,"Inglewood, California"
5,2023,67827,Kansas City Chiefs,Philadelphia Eagles,38,35,5,3,2,4,1,3,State Farm Stadium,"Glendale, Arizona"
6,2024,61629,Kansas City Chiefs,San Francisco 49ers,25,22 (OT),6,4,2,8,5,3,Allegiant Stadium,"Paradise, Nevada"
7,2025,65719,Philadelphia Eagles,Kansas City Chiefs,40,22,5,2,3,7,4,3,Caesars Superdome,"New Orleans, Louisiana"
8,2026,70823,Seattle Seahawks,New England Patriots,29,13,4,2,2,12,6,6,Levi's Stadium,"Santa Clara, California"


In [29]:
merged = pd.merge(sb_clean, sb_2_clean, on='game_year_id', how='outer')

merged['home_team_won'] = merged['home_team'] == merged['winner_team']

merged['home_score'] = merged.apply(
    lambda r: r['winner_score'] if r['home_team_won'] else r['loser_score'], axis=1)
merged['away_score'] = merged.apply(
    lambda r: r['loser_score'] if r['home_team_won'] else r['winner_score'], axis=1)

for stat in ['appearance_no', 'wins_to_date', 'losses_to_date']:
    merged[f'home_team_{stat}'] = merged.apply(
        lambda r: r[f'winner_{stat}'] if r['home_team_won'] else r[f'loser_{stat}'], axis=1)
    merged[f'away_team_{stat}'] = merged.apply(
        lambda r: r[f'loser_{stat}'] if r['home_team_won'] else r[f'winner_{stat}'], axis=1)

merged = merged.drop(columns=[
    'winner_team', 'loser_team', 'winner_score', 'loser_score',
    'winner_appearance_no', 'winner_wins_to_date', 'winner_losses_to_date',
    'loser_appearance_no', 'loser_wins_to_date', 'loser_losses_to_date'
])

merged

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name,Attendance,venue,city,home_team_won,home_score,away_score,home_team_appearance_no,away_team_appearance_no,home_team_wins_to_date,away_team_wins_to_date,home_team_losses_to_date,away_team_losses_to_date
0,2018,Philadelphia Eagles,New England Patriots,4.5,48.5,dome,Nick Foles,Tom Brady,67612,U.S. Bank Stadium,"Minneapolis, Minnesota",False,33,41,10,3,5,1,5,2
1,2019,New England Patriots,Los Angeles Rams,-2.0,55.5,closed,Tom Brady,Jared Goff,70081,Mercedes-Benz Stadium,"Atlanta, Georgia",False,3,13,4,11,1,6,3,5
2,2020,San Francisco 49ers,Kansas City Chiefs,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes,62417,Hard Rock Stadium,"Miami Gardens, Florida",True,31,20,3,7,2,5,1,2
3,2021,Kansas City Chiefs,Tampa Bay Buccaneers,-3.0,55.0,outdoors,Patrick Mahomes,Tom Brady,24835,Raymond James Stadium,"Tampa, Florida",True,31,9,2,4,2,2,0,2
4,2022,Los Angeles Rams,Cincinnati Bengals,-4.5,48.5,dome,Matthew Stafford,Joe Burrow,70048,SoFi Stadium,"Inglewood, California",False,20,23,3,5,0,2,3,3
5,2023,Kansas City Chiefs,Philadelphia Eagles,1.0,51.0,closed,Patrick Mahomes,Jalen Hurts,67827,State Farm Stadium,"Glendale, Arizona",False,35,38,4,5,1,3,3,2
6,2024,San Francisco 49ers,Kansas City Chiefs,-1.5,47.0,dome,Brock Purdy,Patrick Mahomes,61629,Allegiant Stadium,"Paradise, Nevada",True,25,22 (OT),6,8,4,5,2,3
7,2025,Kansas City Chiefs,Philadelphia Eagles,-1.5,48.5,dome,Patrick Mahomes,Jalen Hurts,65719,Caesars Superdome,"New Orleans, Louisiana",True,40,22,5,7,2,4,3,3
8,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70823,Levi's Stadium,"Santa Clara, California",False,13,29,12,4,6,2,6,2


In [30]:
# Fix OT Formatting error
# Flag which games went to OT, based on the score columns as they exist now
merged['went_to_ot'] = (
    merged['home_score'].astype(str).str.contains('OT') |
    merged['away_score'].astype(str).str.contains('OT')
)

# Strip the "(OT)" text and convert to real integers
merged['home_score'] = (
    merged['home_score'].astype(str).str.replace(r'\s*\(OT\)', '', regex=True).astype(int)
)
merged['away_score'] = (
    merged['away_score'].astype(str).str.replace(r'\s*\(OT\)', '', regex=True).astype(int)
)

merged.dtypes[['home_score', 'away_score', 'went_to_ot']]

,0
home_score,int64
away_score,int64
went_to_ot,bool


In [31]:
merged

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name,Attendance,venue,...,home_team_won,home_score,away_score,home_team_appearance_no,away_team_appearance_no,home_team_wins_to_date,away_team_wins_to_date,home_team_losses_to_date,away_team_losses_to_date,went_to_ot
0,2018,Philadelphia Eagles,New England Patriots,4.5,48.5,dome,Nick Foles,Tom Brady,67612,U.S. Bank Stadium,...,False,33,41,10,3,5,1,5,2,False
1,2019,New England Patriots,Los Angeles Rams,-2.0,55.5,closed,Tom Brady,Jared Goff,70081,Mercedes-Benz Stadium,...,False,3,13,4,11,1,6,3,5,False
2,2020,San Francisco 49ers,Kansas City Chiefs,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes,62417,Hard Rock Stadium,...,True,31,20,3,7,2,5,1,2,False
3,2021,Kansas City Chiefs,Tampa Bay Buccaneers,-3.0,55.0,outdoors,Patrick Mahomes,Tom Brady,24835,Raymond James Stadium,...,True,31,9,2,4,2,2,0,2,False
4,2022,Los Angeles Rams,Cincinnati Bengals,-4.5,48.5,dome,Matthew Stafford,Joe Burrow,70048,SoFi Stadium,...,False,20,23,3,5,0,2,3,3,False
5,2023,Kansas City Chiefs,Philadelphia Eagles,1.0,51.0,closed,Patrick Mahomes,Jalen Hurts,67827,State Farm Stadium,...,False,35,38,4,5,1,3,3,2,False
6,2024,San Francisco 49ers,Kansas City Chiefs,-1.5,47.0,dome,Brock Purdy,Patrick Mahomes,61629,Allegiant Stadium,...,True,25,22,6,8,4,5,2,3,True
7,2025,Kansas City Chiefs,Philadelphia Eagles,-1.5,48.5,dome,Patrick Mahomes,Jalen Hurts,65719,Caesars Superdome,...,True,40,22,5,7,2,4,3,3,False
8,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70823,Levi's Stadium,...,False,13,29,12,4,6,2,6,2,False


In [32]:
# Manually fill 2026
lx_mask = merged['game_year_id'] == 2026

merged.loc[lx_mask, 'away_team'] = 'Seattle Seahawks'
merged.loc[lx_mask, 'home_team'] = 'New England Patriots'
merged.loc[lx_mask, 'away_qb_name'] = 'Sam Darnold'
merged.loc[lx_mask, 'home_qb_name'] = 'Drake Maye'
merged.loc[lx_mask, 'roof'] = 'outdoors'
merged.loc[lx_mask, 'spread_line'] = -4.5
merged.loc[lx_mask, 'total_line'] = 45.5

# recompute home_team_won explicitly now that home_team is real, rather than trust the earlier NaN coincidence
merged.loc[lx_mask, 'home_team_won'] = merged.loc[lx_mask, 'home_score'] > merged.loc[lx_mask, 'away_score']

merged.tail(1)

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name,Attendance,venue,...,home_team_won,home_score,away_score,home_team_appearance_no,away_team_appearance_no,home_team_wins_to_date,away_team_wins_to_date,home_team_losses_to_date,away_team_losses_to_date,went_to_ot
8,2026,Seattle Seahawks,New England Patriots,-4.5,45.5,outdoors,Sam Darnold,Drake Maye,70823,Levi's Stadium,...,False,13,29,12,4,6,2,6,2,False


In [33]:
merged

,game_year_id,away_team,home_team,spread_line,total_line,roof,away_qb_name,home_qb_name,Attendance,venue,...,home_team_won,home_score,away_score,home_team_appearance_no,away_team_appearance_no,home_team_wins_to_date,away_team_wins_to_date,home_team_losses_to_date,away_team_losses_to_date,went_to_ot
0,2018,Philadelphia Eagles,New England Patriots,4.5,48.5,dome,Nick Foles,Tom Brady,67612,U.S. Bank Stadium,...,False,33,41,10,3,5,1,5,2,False
1,2019,New England Patriots,Los Angeles Rams,-2.0,55.5,closed,Tom Brady,Jared Goff,70081,Mercedes-Benz Stadium,...,False,3,13,4,11,1,6,3,5,False
2,2020,San Francisco 49ers,Kansas City Chiefs,1.5,52.5,outdoors,Jimmy Garoppolo,Patrick Mahomes,62417,Hard Rock Stadium,...,True,31,20,3,7,2,5,1,2,False
3,2021,Kansas City Chiefs,Tampa Bay Buccaneers,-3.0,55.0,outdoors,Patrick Mahomes,Tom Brady,24835,Raymond James Stadium,...,True,31,9,2,4,2,2,0,2,False
4,2022,Los Angeles Rams,Cincinnati Bengals,-4.5,48.5,dome,Matthew Stafford,Joe Burrow,70048,SoFi Stadium,...,False,20,23,3,5,0,2,3,3,False
5,2023,Kansas City Chiefs,Philadelphia Eagles,1.0,51.0,closed,Patrick Mahomes,Jalen Hurts,67827,State Farm Stadium,...,False,35,38,4,5,1,3,3,2,False
6,2024,San Francisco 49ers,Kansas City Chiefs,-1.5,47.0,dome,Brock Purdy,Patrick Mahomes,61629,Allegiant Stadium,...,True,25,22,6,8,4,5,2,3,True
7,2025,Kansas City Chiefs,Philadelphia Eagles,-1.5,48.5,dome,Patrick Mahomes,Jalen Hurts,65719,Caesars Superdome,...,True,40,22,5,7,2,4,3,3,False
8,2026,Seattle Seahawks,New England Patriots,-4.5,45.5,outdoors,Sam Darnold,Drake Maye,70823,Levi's Stadium,...,False,13,29,12,4,6,2,6,2,False


In [34]:
# Save file
merged.to_csv('superbowl_games_merged.csv', index=False)

from google.colab import files
files.download('superbowl_games_merged.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
tf = pd.read_csv('/content/drive/MyDrive/target_variable_FINAL.csv')

In [36]:
# fix target dataset a bit

# 1) Platform label fix
tf['platform'] = tf['platform'].replace('TheStreet/StubHub', 'StubHub')

# 2) Bucket days_out_approx, with an explicit bucket for the 2 missing rows
def bucket_days(row):
    if pd.isna(row['days_out_approx']):
        return 'final_unspecified'
    d = row['days_out_approx']
    if d <= 1:
        return '0-1_days'
    elif d <= 5:
        return '2-5_days'
    elif d <= 10:
        return '6-10_days'
    else:
        return '11-14_days'

tf['days_out_bucket'] = tf.apply(bucket_days, axis=1)

# 3) Resolve same-platform/same-day conflicts by averaging, flagged
conflict_mask = tf.duplicated(subset=['game_id','platform','days_out_approx'], keep=False) & tf['days_out_approx'].notna()
tf['source_disagreement'] = conflict_mask

tf_resolved = (
    tf.groupby(['game_id','year','kickoff_date','platform','days_out_bucket'], dropna=False, as_index=False)
      .agg(price_usd=('price_usd','mean'),
           source_disagreement=('source_disagreement','any'),
           days_out_approx=('days_out_approx','mean'))
)

tf_resolved

,game_id,year,kickoff_date,platform,days_out_bucket,price_usd,source_disagreement,days_out_approx
0,Super Bowl LII,2018,2018-02-04,SeatGeek,11-14_days,5373.0,False,13.0
1,Super Bowl LIII,2019,2019-02-03,SeatGeek,11-14_days,4993.0,True,13.0
2,Super Bowl LIII,2019,2019-02-03,StubHub,0-1_days,4380.0,False,1.0
3,Super Bowl LIII,2019,2019-02-03,StubHub,2-5_days,4636.0,False,5.0
4,Super Bowl LIV,2020,2020-02-02,SeatGeek,0-1_days,9031.0,False,0.0
5,Super Bowl LIV,2020,2020-02-02,SeatGeek,11-14_days,6870.5,True,13.0
6,Super Bowl LIX,2025,2025-02-09,StubHub,2-5_days,7299.0,False,3.0
7,Super Bowl LIX,2025,2025-02-09,TickPick,2-5_days,6552.0,False,3.0
8,Super Bowl LV,2021,2021-02-07,SeatGeek,11-14_days,10224.5,True,13.0
9,Super Bowl LV,2021,2021-02-07,TicketIQ,final_unspecified,12477.0,False,NaN


In [37]:
# --- Platform encoding (one-hot) ---
platform_dummies = pd.get_dummies(tf_resolved['platform'], prefix='platform')
tf_features = pd.concat([tf_resolved, platform_dummies], axis=1)

# --- Price normalization per game ---
# anchor = each game's earliest known price observation (highest days_out_approx)
anchor = (
    tf_features.loc[tf_features.groupby('game_id')['days_out_approx'].idxmax()]
    [['game_id', 'price_usd']]
    .rename(columns={'price_usd': 'anchor_price'})
)

tf_features = tf_features.merge(anchor, on='game_id', how='left')
tf_features['price_pct_of_anchor'] = tf_features['price_usd'] / tf_features['anchor_price'] * 100

tf_features

,game_id,year,kickoff_date,platform,days_out_bucket,price_usd,source_disagreement,days_out_approx,platform_SeatGeek,platform_StubHub,platform_TickPick,platform_TicketIQ,anchor_price,price_pct_of_anchor
0,Super Bowl LII,2018,2018-02-04,SeatGeek,11-14_days,5373.0,False,13.0,True,False,False,False,5373.0,100.000000
1,Super Bowl LIII,2019,2019-02-03,SeatGeek,11-14_days,4993.0,True,13.0,True,False,False,False,4993.0,100.000000
2,Super Bowl LIII,2019,2019-02-03,StubHub,0-1_days,4380.0,False,1.0,False,True,False,False,4993.0,87.722812
3,Super Bowl LIII,2019,2019-02-03,StubHub,2-5_days,4636.0,False,5.0,False,True,False,False,4993.0,92.849990
4,Super Bowl LIV,2020,2020-02-02,SeatGeek,0-1_days,9031.0,False,0.0,True,False,False,False,6870.5,131.446037
5,Super Bowl LIV,2020,2020-02-02,SeatGeek,11-14_days,6870.5,True,13.0,True,False,False,False,6870.5,100.000000
6,Super Bowl LIX,2025,2025-02-09,StubHub,2-5_days,7299.0,False,3.0,False,True,False,False,7299.0,100.000000
7,Super Bowl LIX,2025,2025-02-09,TickPick,2-5_days,6552.0,False,3.0,False,False,True,False,7299.0,89.765721
8,Super Bowl LV,2021,2021-02-07,SeatGeek,11-14_days,10224.5,True,13.0,True,False,False,False,10224.5,100.000000
9,Super Bowl LV,2021,2021-02-07,TicketIQ,final_unspecified,12477.0,False,NaN,False,False,False,True,10224.5,122.030417


In [38]:
# Align the join key
merged_renamed = merged.rename(columns={'game_year_id': 'year'})

# Left join: every price-observation row picks up its game's context columns
model_input = tf_features.merge(merged_renamed, on='year', how='left')

model_input

,game_id,year,kickoff_date,platform,days_out_bucket,price_usd,source_disagreement,days_out_approx,platform_SeatGeek,platform_StubHub,...,home_team_won,home_score,away_score,home_team_appearance_no,away_team_appearance_no,home_team_wins_to_date,away_team_wins_to_date,home_team_losses_to_date,away_team_losses_to_date,went_to_ot
0,Super Bowl LII,2018,2018-02-04,SeatGeek,11-14_days,5373.0,False,13.0,True,False,...,False,33,41,10,3,5,1,5,2,False
1,Super Bowl LIII,2019,2019-02-03,SeatGeek,11-14_days,4993.0,True,13.0,True,False,...,False,3,13,4,11,1,6,3,5,False
2,Super Bowl LIII,2019,2019-02-03,StubHub,0-1_days,4380.0,False,1.0,False,True,...,False,3,13,4,11,1,6,3,5,False
3,Super Bowl LIII,2019,2019-02-03,StubHub,2-5_days,4636.0,False,5.0,False,True,...,False,3,13,4,11,1,6,3,5,False
4,Super Bowl LIV,2020,2020-02-02,SeatGeek,0-1_days,9031.0,False,0.0,True,False,...,True,31,20,3,7,2,5,1,2,False
5,Super Bowl LIV,2020,2020-02-02,SeatGeek,11-14_days,6870.5,True,13.0,True,False,...,True,31,20,3,7,2,5,1,2,False
6,Super Bowl LIX,2025,2025-02-09,StubHub,2-5_days,7299.0,False,3.0,False,True,...,True,40,22,5,7,2,4,3,3,False
7,Super Bowl LIX,2025,2025-02-09,TickPick,2-5_days,6552.0,False,3.0,False,False,...,True,40,22,5,7,2,4,3,3,False
8,Super Bowl LV,2021,2021-02-07,SeatGeek,11-14_days,10224.5,True,13.0,True,False,...,True,31,9,2,4,2,2,0,2,False
9,Super Bowl LV,2021,2021-02-07,TicketIQ,final_unspecified,12477.0,False,NaN,False,False,...,True,31,9,2,4,2,2,0,2,False


In [39]:
model_input.dtypes

,0
game_id,object
year,int64
kickoff_date,object
platform,object
days_out_bucket,object
price_usd,float64
source_disagreement,bool
days_out_approx,float64
platform_SeatGeek,bool
platform_StubHub,bool


In [40]:
# 4) Drop venue/city
model_input = model_input.drop(columns=['venue','city'])

# 1) Encode roof
roof_dummies = pd.get_dummies(model_input['roof'], prefix='roof')
model_input = pd.concat([model_input, roof_dummies], axis=1)

# 2) Encode team names
away_team_dummies = pd.get_dummies(model_input['away_team'], prefix='away_team')
home_team_dummies = pd.get_dummies(model_input['home_team'], prefix='home_team')
model_input = pd.concat([model_input, away_team_dummies, home_team_dummies], axis=1)

# 3) QB "repeat appearance within our dataset window" boolean
model_input = model_input.sort_values('year').reset_index(drop=True)
seen_qbs = set()
home_repeat, away_repeat = [], []
for _, row in model_input.drop_duplicates('game_id').iterrows():
    home_repeat.append(row['home_qb_name'] in seen_qbs)
    away_repeat.append(row['away_qb_name'] in seen_qbs)
    seen_qbs.add(row['home_qb_name'])
    seen_qbs.add(row['away_qb_name'])
qb_repeat_map = pd.DataFrame({
    'game_id': model_input.drop_duplicates('game_id')['game_id'],
    'home_qb_repeat_appearance': home_repeat,
    'away_qb_repeat_appearance': away_repeat
})
model_input = model_input.merge(qb_repeat_map, on='game_id', how='left')

# 5) Derived features
model_input['point_margin'] = model_input['home_score'] - model_input['away_score']
model_input['blowout'] = model_input['point_margin'].abs() >= 15
model_input['underdog_won'] = (
    (model_input['home_team_won'] & (model_input['spread_line'] < 0)) |
    (~model_input['home_team_won'] & (model_input['spread_line'] > 0))
)

model_input

,game_id,year,kickoff_date,platform,days_out_bucket,price_usd,source_disagreement,days_out_approx,platform_SeatGeek,platform_StubHub,...,home_team_Kansas City Chiefs,home_team_Los Angeles Rams,home_team_New England Patriots,home_team_Philadelphia Eagles,home_team_Tampa Bay Buccaneers,home_qb_repeat_appearance,away_qb_repeat_appearance,point_margin,blowout,underdog_won
0,Super Bowl LII,2018,2018-02-04,SeatGeek,11-14_days,5373.0,False,13.0,True,False,...,False,False,True,False,False,False,False,-8,False,True
1,Super Bowl LIII,2019,2019-02-03,SeatGeek,11-14_days,4993.0,True,13.0,True,False,...,False,True,False,False,False,False,True,-10,False,False
2,Super Bowl LIII,2019,2019-02-03,StubHub,0-1_days,4380.0,False,1.0,False,True,...,False,True,False,False,False,False,True,-10,False,False
3,Super Bowl LIII,2019,2019-02-03,StubHub,2-5_days,4636.0,False,5.0,False,True,...,False,True,False,False,False,False,True,-10,False,False
4,Super Bowl LIV,2020,2020-02-02,SeatGeek,0-1_days,9031.0,False,0.0,True,False,...,True,False,False,False,False,False,False,11,False,False
5,Super Bowl LIV,2020,2020-02-02,SeatGeek,11-14_days,6870.5,True,13.0,True,False,...,True,False,False,False,False,False,False,11,False,False
6,Super Bowl LV,2021,2021-02-07,TicketIQ,final_unspecified,12477.0,False,NaN,False,False,...,False,False,False,False,True,True,True,22,True,True
7,Super Bowl LV,2021,2021-02-07,SeatGeek,11-14_days,10224.5,True,13.0,True,False,...,False,False,False,False,True,True,True,22,True,True
8,Super Bowl LVI,2022,2022-02-13,SeatGeek,6-10_days,8102.0,False,6.0,True,False,...,False,False,False,False,False,False,False,-3,False,False
9,Super Bowl LVI,2022,2022-02-13,StubHub,final_unspecified,8869.0,False,NaN,False,True,...,False,False,False,False,False,False,False,-3,False,False


In [41]:
# Save file
model_input.to_csv('final_model_output.csv', index=False)

from google.colab import files
files.download('final_model_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
model_input.columns.tolist()

['game_id',
 'year',
 'kickoff_date',
 'platform',
 'days_out_bucket',
 'price_usd',
 'source_disagreement',
 'days_out_approx',
 'platform_SeatGeek',
 'platform_StubHub',
 'platform_TickPick',
 'platform_TicketIQ',
 'anchor_price',
 'price_pct_of_anchor',
 'away_team',
 'home_team',
 'spread_line',
 'total_line',
 'roof',
 'away_qb_name',
 'home_qb_name',
 'Attendance',
 'home_team_won',
 'home_score',
 'away_score',
 'home_team_appearance_no',
 'away_team_appearance_no',
 'home_team_wins_to_date',
 'away_team_wins_to_date',
 'home_team_losses_to_date',
 'away_team_losses_to_date',
 'went_to_ot',
 'roof_closed',
 'roof_dome',
 'roof_outdoors',
 'away_team_Kansas City Chiefs',
 'away_team_Los Angeles Rams',
 'away_team_New England Patriots',
 'away_team_Philadelphia Eagles',
 'away_team_San Francisco 49ers',
 'away_team_Seattle Seahawks',
 'home_team_Cincinnati Bengals',
 'home_team_Kansas City Chiefs',
 'home_team_Los Angeles Rams',
 'home_team_New England Patriots',
 'home_team_Phila

In [43]:
# --- Build model input table: target + approved features (rank 1-6) ---

target_col = 'price_usd'

feature_cols = [
    'days_out_approx',
    'platform_SeatGeek', 'platform_StubHub', 'platform_TickPick', 'platform_TicketIQ',
    'home_qb_repeat_appearance', 'away_qb_repeat_appearance',
    'home_team_appearance_no', 'away_team_appearance_no',
    'spread_line'
]

group_col = 'game_id'  # kept for LOGO-CV grouping, not a model feature

model_input_v1 = model_input[[group_col, target_col] + feature_cols].copy()

# Sanity checks
print("Shape:", model_input_v1.shape)  # expect (21, 12)
print("\nMissing values per column:")
print(model_input_v1.isnull().sum())
print("\nDtypes:")
print(model_input_v1.dtypes)

model_input_v1.head()

Shape: (21, 12)

Missing values per column:
game_id                      0
price_usd                    0
days_out_approx              2
platform_SeatGeek            0
platform_StubHub             0
platform_TickPick            0
platform_TicketIQ            0
home_qb_repeat_appearance    0
away_qb_repeat_appearance    0
home_team_appearance_no      0
away_team_appearance_no      0
spread_line                  0
dtype: int64

Dtypes:
game_id                       object
price_usd                    float64
days_out_approx              float64
platform_SeatGeek               bool
platform_StubHub                bool
platform_TickPick               bool
platform_TicketIQ               bool
home_qb_repeat_appearance       bool
away_qb_repeat_appearance       bool
home_team_appearance_no        int64
away_team_appearance_no        int64
spread_line                  float64
dtype: object


,game_id,price_usd,days_out_approx,platform_SeatGeek,platform_StubHub,platform_TickPick,platform_TicketIQ,home_qb_repeat_appearance,away_qb_repeat_appearance,home_team_appearance_no,away_team_appearance_no,spread_line
0,Super Bowl LII,5373.0,13.0,True,False,False,False,False,False,10,3,4.5
1,Super Bowl LIII,4993.0,13.0,True,False,False,False,False,True,4,11,-2.0
2,Super Bowl LIII,4380.0,1.0,False,True,False,False,False,True,4,11,-2.0
3,Super Bowl LIII,4636.0,5.0,False,True,False,False,False,True,4,11,-2.0
4,Super Bowl LIV,9031.0,0.0,True,False,False,False,False,False,3,7,1.5


In [48]:
model_input_v1['days_out_approx'] = model_input_v1['days_out_approx'].fillna(0)

In [49]:
# Save file
model_input_v1.to_csv('final_model_output_small.csv', index=False)

from google.colab import files
files.download('final_model_output_small.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>